# QRNG-Based Spacecraft Navigation — Interactive Notebook

This notebook runs a compact version of the QRNG demo:

- Generate a circular orbit (ground truth)
- Get random bits (ANU QRNG → OS fallback → PRNG)
- Convert bits → uniform → Gaussian noise
- Inject noise + drift into simulated sensor data
- Estimate trajectory using a simple 4-state Kalman filter

Use small `n_steps` (e.g., 150) for fast runs.


In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root (only needed if VSCode kernel is not auto-detected)
ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from sim.orbit import OrbitSimulator
from estimator.kalman import SimpleKalman
from extras.qrng.qrng_utils import generate_random_bits

# ---- helper converters ----

def bits_to_floats(bits, block_size=16):
    n_blocks = len(bits) // block_size
    arr = bits[:n_blocks*block_size].reshape((n_blocks, block_size))
    vals = []
    for row in arr:
        v = 0
        for i, b in enumerate(row):
            v |= (int(b) << i)
        vals.append(v)
    vals = np.array(vals, dtype=np.uint32)
    maxv = 2**block_size - 1
    return (vals / maxv) * 2 - 1   # map to [-1,1]

def uniform_to_gaussian(u):
    u = np.clip((u + 1) / 2, 1e-12, 1 - 1e-12)
    u1 = u[0::2];  u2 = u[1::2]
    r = np.sqrt(-2 * np.log(u1))
    theta = 2 * np.pi * u2
    z0 = r * np.cos(theta)
    z1 = r * np.sin(theta)
    return np.hstack((z0, z1))


In [ ]:
# ---- Parameters ----
n_steps = 150
dt = 10.0
use_qrng = False      # True → try ANU QRNG
noise_std = 0.001
drift_scale = 0.00005

# ---- Orbit ----
sim = OrbitSimulator()
t, true_pos, true_vel = sim.simulate_circular(n_steps=n_steps, dt=dt)

# ---- QRNG bits ----
bits_needed = 16 * (2 * n_steps)
bits = generate_random_bits(bits_needed, mode='anu' if use_qrng else 'os')
floats = bits_to_floats(bits)
gauss = uniform_to_gaussian(floats)
noise = gauss[:2*n_steps].reshape(n_steps, 2)

# ---- Measurements ----
drift = np.cumsum(noise * drift_scale, axis=0)
meas = true_pos + noise * noise_std + drift

# ---- Kalman Filter ----
kf = SimpleKalman(dt=dt)
x0 = np.array([meas[0,0], meas[0,1], true_vel[0,0], true_vel[0,1]])
kf.set_initial(x0)

est = np.zeros((n_steps, 4))
for i in range(n_steps):
    est[i] = kf.step(meas[i])

rmse = np.sqrt(np.mean((est[:, :2] - true_pos)**2, axis=1))

# ---- Plot ----
plt.figure(figsize=(10, 4))

plt.subplot(1,2,1)
plt.plot(true_pos[:,0], true_pos[:,1], label="True", color="#1f77b4")
plt.scatter(meas[:,0], meas[:,1], s=5, alpha=0.6, label="Measured", color="#ff7f0e")
plt.plot(est[:,0], est[:,1], "--", label="Estimated", color="#2ca02c")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.title("Trajectory (km)")

plt.subplot(1,2,2)
plt.plot(t, rmse, color="#d62728")
plt.xlabel("Time (s)")
plt.ylabel("Position RMSE (km)")
plt.title("RMSE over time")

plt.tight_layout()
plt.show()

# ---- Save Outputs ----
os.makedirs("outputs", exist_ok=True)

df = pd.DataFrame({
    "t": t,
    "true_x": true_pos[:,0], "true_y": true_pos[:,1],
    "meas_x": meas[:,0], "meas_y": meas[:,1],
    "est_x": est[:,0], "est_y": est[:,1],
    "rmse": rmse
})

df.to_csv("outputs/results_from_notebook.csv", index=False)
plt.savefig("outputs/trajectory_rmse_notebook.png", dpi=200)

print("Saved: outputs/results_from_notebook.csv")
print("Saved: outputs/trajectory_rmse_notebook.png")
